# Crowd Head-ID Tracking — Colab T4 Handoff (v7)

**Setup:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

This notebook reproduces the current best head-ID stability result (arm `fix_0p16_wide_guard_app`, "v7") on a Colab **T4**. It is fully self-contained:

1. Clones the pipeline code from GitHub (branch `feat/hybrid-detection`).
2. Downloads the fine-tuned head detector + sample video from the `v7-handoff` GitHub release.
3. Runs `scripts/run_head_id_stability.py` for the `baseline` and `fix_0p16_wide_guard_app` arms.
4. Shows a metrics comparison and lets you download the annotated videos.

Expected on a T4: **~6–10 min per arm** for the 803-frame clip. Local reference numbers: baseline ≈ 338 unique IDs vs **v7 ≈ 125 unique @ 3.1× inflation** (lower = better; the model and config are identical to the A100 run that scored 105 @ 2.5×).

In [ ]:
# 1) Confirm a GPU is attached (must show Tesla T4 or similar)
import torch
!nvidia-smi -L
assert torch.cuda.is_available(), "No GPU! Set Runtime > Change runtime type > T4 GPU, then Run all."
print("CUDA device:", torch.cuda.get_device_name(0))

In [ ]:
# 2) Get the pipeline code (branch feat/hybrid-detection)
import os
REPO_URL = "https://github.com/Sriniketh24/Crowd-analysis.git"
BRANCH = "feat/hybrid-detection"
%cd /content
if not os.path.exists("/content/Crowd-analysis"):
    !git clone --depth 1 --branch {BRANCH} {REPO_URL}
else:
    !cd /content/Crowd-analysis && git fetch --depth 1 origin {BRANCH} && git checkout {BRANCH} && git reset --hard origin/{BRANCH}
%cd /content/Crowd-analysis
!git log -1 --oneline

In [ ]:
# 3) Install pinned dependencies.
# cv2/numpy/pandas are preinstalled on Colab; torch stays as Colab's CUDA build.
# If pip complains about the exact pins, relax to:  ultralytics>=8.4 supervision>=0.28
%pip install -q "ultralytics==8.4.58" "supervision==0.28.0" "tqdm>=4.66"
import ultralytics, supervision
print("ultralytics", ultralytics.__version__, "| supervision", supervision.__version__)

In [ ]:
# 4) Download the fine-tuned head detector + sample video from the v7-handoff release
import os, urllib.request, pathlib
BASE = "https://github.com/Sriniketh24/Crowd-analysis/releases/download/v7-handoff"
assets = {
    "models/fine_tuned/head_detector_s/weights/best.pt": f"{BASE}/head_detector_s_best.pt",
    "data/input_videos/sample.mp4": f"{BASE}/sample.mp4",
}
for dest, url in assets.items():
    pathlib.Path(dest).parent.mkdir(parents=True, exist_ok=True)
    if not os.path.exists(dest) or os.path.getsize(dest) == 0:
        print("Downloading", url)
        urllib.request.urlretrieve(url, dest)
    print(f"{dest}: {os.path.getsize(dest)/1e6:.1f} MB")

In [ ]:
# 5) Run the v7 pipeline (baseline vs fix_0p16_wide_guard_app) on the GPU.
# To run ONLY the winner (about half the time), set --arms fix_0p16_wide_guard_app
!python scripts/run_head_id_stability.py --video data/input_videos/sample.mp4 --model models/fine_tuned/head_detector_s/weights/best.pt --arms baseline fix_0p16_wide_guard_app --device cuda --half --imgsz 1536 --output-root data/outputs/handoff_run

In [ ]:
# 6) Show the metrics comparison (lower confirmed_unique / inflation = better)
import pandas as pd
m = pd.read_csv("data/outputs/handoff_run/sample/rpee_head_s/comparison_metrics.csv")
cols = ["arm", "confirmed_unique", "inflation_factor", "peak_concurrent_confirmed",
        "residual_switch_events", "duplicate_id_frames", "gt_mae"]
m[cols]

In [ ]:
# 7) Download the annotated videos (full-res ~29 MB each)
from google.colab import files
OUT = "data/outputs/handoff_run/sample/rpee_head_s"
v7 = f"{OUT}/fix_0p16_wide_guard_app_annotated.mp4"
print("v7 annotated video:", v7)
files.download(v7)
# Side-by-side baseline-vs-v7 comparison (uncomment to also download):
# files.download(f"{OUT}/side_by_side.mp4")

## Optional: save outputs to Google Drive (persists after the session ends)
Colab `/content` is wiped when the runtime recycles. Run the cell below to copy results to your Drive.

In [ ]:
# (Optional) Persist outputs to Drive
from google.colab import drive
import shutil, os
drive.mount("/content/drive")
dst = "/content/drive/MyDrive/crowd-analysis-outputs/handoff_run"
os.makedirs(dst, exist_ok=True)
shutil.copytree("data/outputs/handoff_run", dst, dirs_exist_ok=True)
print("Copied to", dst)

## Notes & troubleshooting

- **Run a different / your own arm:** all available arm names are in `DEFAULT_ARMS` inside `scripts/run_head_id_stability.py`. The winner is `fix_0p16_wide_guard_app`.
- **Your own video:** upload it, change `--video` to its path. Note the `gt_mae` metric is calibrated to `sample.mp4` only (hand-counted ground truth `MANUAL_GT` in the script) — it will be meaningless on a different video unless you update that dict.
- **Speed:** at `--imgsz 1536` the detector is heavy. To go faster for a quick check, lower to `--imgsz 1280` (slightly fewer far heads) or add `--max-frames 200`.
- **No GPU / wrong runtime:** cell 1 will assert. Fix via `Runtime → Change runtime type → T4 GPU`.
- **pip pin errors:** relax the pins in cell 3 to `ultralytics>=8.4 supervision>=0.28`.
- The tracker uses `supervision.ByteTrack`; if `supervision` is missing it silently falls back to a weaker greedy-IoU tracker, so keep cell 3 working.